DOGS-CATS CLASSIFICATION MODEL

In [1]:
# Standard library
import copy
import glob
import multiprocessing
import os
import time
import zipfile

# Pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# Related third party
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

c:\FHDO\Research Thesis\project\embeddedNeuralNetwork\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [3]:
def print_model_size(mdl):
    torch.save(mdl.state_dict(), "tmp.pt")
    print("%.2f MB" %(os.path.getsize("tmp.pt")/1e6))
    os.remove('tmp.pt')

In [4]:
input_size = (224,224)
mean = [0.485, 0.456, 0.406] 
std = [0.229, 0.224, 0.225]
transform = transforms.Compose([
    transforms.Resize(input_size),  # Resize to a fixed size
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [5]:
class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for label, folder_name in enumerate(['Dog', 'Cat']):
            folder_path = os.path.join(self.root_dir, folder_name)
            for file_name in os.listdir(folder_path):
                file_path = os.path.join(folder_path, file_name)
                
                try:
                    with Image.open(file_path) as img:
                        
                        if img.mode != 'RGB':
                            img = img.convert('RGB')
                        
                        if img.mode != 'RGB':
                            print(f"Skipping {file_path} because it does not have 3 channels (RGB)")
                            continue

                        self.image_paths.append(file_path)
                        self.labels.append(label)
                        
                except Exception as e:
                    print(f"Skipping {file_path} due to error: {e}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        
        with Image.open(image_path) as img:

            if img.mode != 'RGB':
                img = img.convert('RGB')

            if self.transform:
                img = self.transform(img)
            
        return img, label

In [6]:
dataset = CustomDataset(root_dir='../data/PetImages', transform=transform)

# Calculate split sizes
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

# Split dataset into train and test
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

Skipping ../data/PetImages\Dog\11702.jpg due to error: cannot identify image file '../data/PetImages\\Dog\\11702.jpg'


c:\FHDO\Research Thesis\project\embeddedNeuralNetwork\venv\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Skipping ../data/PetImages\Dog\Thumbs.db due to error: cannot identify image file '../data/PetImages\\Dog\\Thumbs.db'
Skipping ../data/PetImages\Cat\666.jpg due to error: cannot identify image file '../data/PetImages\\Cat\\666.jpg'
Skipping ../data/PetImages\Cat\Thumbs.db due to error: cannot identify image file '../data/PetImages\\Cat\\Thumbs.db'


In [7]:
# Create DataLoader for train and test sets
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
for images, labels in train_loader:
    print(images.shape)
    print(labels.shape)
    break

torch.Size([128, 3, 224, 224])
torch.Size([128])


In [8]:
def train_epoch(model, criterion, optimizer, data_loader, device,epoch):
    model.train()
    
    epoch_loss = 0.0
    num_batches = len(data_loader)
    
    for batch_idx, (image, target) in enumerate(tqdm(data_loader)):
        image, target = image.to(device), target.to(device)
        
        output = model(image)
        
        loss = criterion(output, target)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    # Calculate average epoch loss
    avg_epoch_loss = epoch_loss / num_batches
    print(f"Epoch = {epoch+1} || Training Loss: {avg_epoch_loss:.4f}")


def evaluate(model, criterion, data_loader, device,epoch):
    
    model.eval()
    
    epoch_loss = 0.0
    
    correct_predictions = 0
    total_predictions = 0
    
    num_batches = len(data_loader)
    
    with torch.no_grad():
       
        for image, target in tqdm(data_loader):
            image, target = image.to(device), target.to(device)
            output = model(image)
            loss = criterion(output, target)
            # Accumulate batch loss
            epoch_loss += loss.item()
            
            # Calculate accuracy
            _, predicted = torch.max(output, 1)  # Get the predicted class index
            correct_predictions += (predicted == target).sum().item()
            total_predictions += target.size(0)
            
    # Calculate average epoch loss
    avg_epoch_loss = epoch_loss / num_batches
    accuracy = correct_predictions / total_predictions
    
    print(f"Epoch = {epoch+1} || Test Loss: {avg_epoch_loss:.4f} || Test Accuracy: {accuracy:.4f}")

In [9]:
class MobileNet(torch.nn.Module):
    def __init__(self):
        super(MobileNet, self).__init__()
        self.model = models.mobilenet_v2(weights='MobileNet_V2_Weights.IMAGENET1K_V1')  
        
        # for param in self.model.parameters():
        #     param.requires_grad = False
            
        
        
        self.model.classifier[1] = nn.Sequential(
            nn.Linear(in_features=self.model.classifier[1].in_features,out_features=512),
            nn.LeakyReLU(negative_slope=0.02,inplace=False),
            nn.BatchNorm1d(num_features=512),
            nn.Dropout(p=0.4,inplace=False),
            nn.Linear(in_features=512,out_features=2),
            nn.Softmax(dim=1))
        
        # print(self.model)

    def forward(self, x):
        x = self.model(x)
        return x

In [50]:
model = MobileNet()
epoch = 5
criterion = nn.CrossEntropyLoss(reduction='mean')
optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001)
for nepoch in range(epoch):
    train_epoch(model, criterion, optimizer, train_loader, device, nepoch)

# Evaluation
print("Evaluating quantized model...")
evaluate(model, criterion, test_loader, device, nepoch)

100%|██████████| 157/157 [26:34<00:00, 10.15s/it]


Epoch = 1 || Training Loss: 0.3403


100%|██████████| 157/157 [26:20<00:00, 10.06s/it]


Epoch = 2 || Training Loss: 0.3219


100%|██████████| 157/157 [27:44<00:00, 10.60s/it]


...

Epoch = 4 || Training Loss: 0.3170


100%|██████████| 157/157 [25:32<00:00,  9.76s/it]


Epoch = 5 || Training Loss: 0.3158
Evaluating quantized model...


100%|██████████| 40/40 [02:19<00:00,  3.48s/it]

Epoch = 5 || Test Loss: 0.3238 || Test Accuracy: 0.9882


In [51]:
# Save model back to Drive
torch.save(model.state_dict(), "../model/original_catndog_mobilenetv2.pth")

In [10]:
# Measure Validate Accuracy and Validate Loss of Orginal model
model = MobileNet()
criterion = nn.CrossEntropyLoss(reduction='mean')
model.load_state_dict(torch.load("../model/original_catndog_mobilenetv2.pth"))
count = 0
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Conv2d):
        count = count + 1
print(f"Number of conv layers: {count}")
model.eval()
print("Evaluating original model...")
evaluate(model, criterion, test_loader, device, 0)

Number of conv layers: 52
Evaluating original model...


100%|██████████| 40/40 [00:50<00:00,  1.27s/it]


Epoch = 1 || Test Loss: 0.3166 || Test Accuracy: 0.9964


In [12]:
print(len(test_loader))
print_model_size(model)

40
11.76 MB
